In [ ]:
# Install required packages
#!pip install -q -U qiskit qiskit-ibm-runtime
#!pip install -q matplotlib numpy ipython
#!pip install -q samplomatic
#!pip install -q qiskit-addon-utils qiskit-addon-pna qiskit-addon-slc
#!pip install --upgrade qc-grader
#!pip install plotly
#!pip install pylatexenc
#!pip install nbformat

In [ ]:
### Scientific Computing
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

### IPython
from IPython.display import display

### Qiskit 
from qiskit import QuantumCircuit
from qiskit.quantum_info import Pauli, SparsePauliOp
from qiskit.transpiler import generate_preset_pass_manager

### Qiskit IBM Runtime
from qiskit_ibm_runtime import QiskitRuntimeService, Executor, QuantumProgram
from qiskit_ibm_runtime.options import EstimatorOptions
from qiskit_ibm_runtime.options_models.noise_learner_v3_options import NoiseLearnerV3Options
from qiskit_ibm_runtime.noise_learner_v3 import NoiseLearnerV3

### Samplomatic
import samplomatic
from samplomatic import Twirl, InjectNoise, ChangeBasis, build
from samplomatic.transpiler import generate_boxing_pass_manager
from samplomatic.utils import find_unique_box_instructions, get_annotation

### Qiskit Addons
from qiskit_addon_utils.exp_vals.measurement_bases import get_measurement_bases
from qiskit_addon_utils.exp_vals.expectation_values import executor_expectation_values
from qiskit_addon_utils.noise_management import trex_factors, gamma_from_noisy_boxes
from qiskit_addon_pna import generate_noise_mitigating_observable
from qiskit_addon_slc.bounds import compute_backward_bounds, compute_forward_bounds, compute_local_scales, merge_bounds
from qiskit_addon_slc.utils import generate_noise_model_paulis, map_modifier_ref_to_ref
from qiskit_addon_slc.visualization import draw_shaded_lightcone


# Grader imports
from qc_grader.challenges.qgss_2026 import (
    grade_lab3_ex1,
    grade_lab3_ex2,
    grade_lab3_ex3,
    grade_lab3_ex4,
    grade_lab3_ex5,
)

# grader function to check your progress
from qc_grader.challenges.qgss_2026 import check_progress

#### Define the backend:

In this lab we will work with a __Heron device__. As stated above, noise learning jobs executed on the backend cannot be executed with a simulator. By focusing on a Heron device, participants working with an Open Plan account can participate in this lab. 

It is a good idea to you use the _same backend_ throughout this tutorial. We will be learning the noise in the backend and using that noise model to implement advanced error mitigation techniques. Therefore, if you change backend halfway through, the learned noise model will not be applicable to the new backend.

Note: If you are having trouble connecting to a backend, return to Lab 0 and make sure you have set up your IBM Quantum Platform Account correctly.

In [ ]:
service = QiskitRuntimeService()

# Define the backend: 
service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True,
    filters=lambda b: b.processor_type.get('family') == 'Heron'
)

print(f"Backend     : {backend.name}")
print(f"# qubits    : {backend.num_qubits}")
print(f"Basis gates : {backend.basis_gates}")

# Create ISA pass manager for transpilation with optimization_level=0
isa_pm = generate_preset_pass_manager(backend=backend, optimization_level=0)


## Chapter 0 — Errors on today's quantum hardware

Today's quantum hardware is noisy. Every gate, every idle moment, every measurement introduces small errors, and at utility scale (circuits on 100+ qubits) those errors accumulate to a degree that can no longer be ignored. Until full fault tolerance arrives, the only way to get *useful* answers out of these devices is to manage that noise, to suppress it where we can, to characterize it where we can't, and to mitigate its bias when we read out a result. Dealing with noise is therefore a central part of getting useful results from today's devices, and the techniques for doing it are an active area of development.

The techniques for handling noise fall into roughly four categories. They differ in *when* they act, in *what* they cost, and in whether they fix errors, flag them, or merely correct the statistics after the fact — so it is worth seeing all four before we introduce the new tools this lab is built around.

- **Error correction (EC)** encodes logical information redundantly, repeatedly extracts syndromes, and when implemented fault-tolerantly, can suppress logical errors below physical error rates. Full QEC is prohibitive on near-term devices because of its qubit and time overheads.

- **Error detection (ED)** flags runs that are likely corrupted and discards them via post-selection, keeping a smaller but cleaner sample. The flag can come from a detection-only QEC code, one whose distance heralds an error without giving enough information to identify and reverse it. In that case, making ED a limiting case of error correction. ED lets a higher-level code in a hierarchical scheme treat the flagged error as an *erasure* (a known location is far easier to correct than an unknown one). It can equally come from cheaper, code-free checks, such as verifying a symmetry the ideal circuit should preserve, e.g. a conserved parity or particle number, and rejecting shots that violate it. Either way, ED trades a reduced usable sample against an implementation overhead well below that of fault-tolerant EC.

- **Error mitigation (EM)** does not fix individual shots at all. It biases or re-weights the statistics across many noisy runs so that the *estimate* of an expectation value (or distribution) is closer to the ideal one ([Cai et al., *Quantum error mitigation*, Rev. Mod. Phys. 95, 2023](https://journals.aps.org/rmp/abstract/10.1103/RevModPhys.95.045005)). This is the regime where today's utility-scale experiments operate, and it is the focus of this lab.

- **Error suppression (ES)** acts during execution to reshape the noise itself. Examples include dynamical decoupling (DD), which inserts pulses on idle qubits to cancel coherent error, and Pauli twirling (PT), which randomizes gate dressings so that the residual error becomes a stochastic Pauli channel ([Wallman & Emerson, PRA 94, 2016](https://journals.aps.org/pra/abstract/10.1103/PhysRevA.94.052325)).

Two of these four you have already used, whether you noticed or not, if you have run the Qiskit Runtime primitives. Error suppression and error mitigation are what `Estimator` implements (error correction and detection are not something the primitive applies for you). At its default option, `Estimator` already corrects for systematic readout bias (Twirled Readout Error eXtinction, TREX). You can also go further and enable additional error suppression and error mitigation methods yourself using `EstimatorOptions` as we shall see in section 1.1.2. What all of these options have in common is that they act on your circuit as a single _circuit-wide_ policy, which adds a fixed kind of overhead to every run. 

This lab introduces the tools (_Samplomatic_) that you can use alongside Qiskit Runtime to go beyond that _circuit-wide_ setting, allowing you to manage noise layer by layer and observable by observable. In Chapter 1, we begin by giving an overview of what the familiar `EstimatorOptions` do and what they cost.

## Chapter 1 — Error mitigation as a Runtime product, and the theory behind its cost

This chapter takes the familiar Runtime options and opens them up:
- Section 1.1 walks through the error suppression and error mitigation methods `Estimator` offers and has you configure each one by hand. 

- Section 1.2 then steps one level *below* the switches to the small piece of algebra — *dressing* and *commutation* — that is the most basic principle these techniques rest on. 

- Section 1.3 closes the chapter by asking what these whole-circuit switches *cannot* do, and what they cost in terms of resource overhead. This motivates the finer, box-by-box tools the rest of the lab is built around.

## 1.1  Error mitigation as a cloud service

The introduction noted that Qiskit Runtime exposes error mitigation as a product: a set of options you turn on, applied to your circuit as a single circuit-wide policy. This section catalogs those options on the `EstimatorV2` primitive — what each one does, and which `resilience_level` preset turns it on (section 1.1.1) — and then looks at each method's settings and behavior in detail (section 1.1.2). It closes with Exercise 1, in which you configure each of the five methods on its own `EstimatorOptions` so the effect of each one stays separable. The algebra these methods rest on comes in section 1.2.

### 1.1.1  The options on the Runtime primitives

The `EstimatorV2` primitive bundles these error suppression and error mitigation techniques behind a single `EstimatorOptions` object. The official catalogue is [Error mitigation and suppression techniques](https://quantum.cloud.ibm.com/docs/en/guides/error-mitigation-and-suppression-techniques). The `resilience_level` option is a convenience preset that turns a fixed set of them on at once. The table below shows each technique, its role, and which preset level enables it. We will introduce the exact attribute paths in section 1.1.2.

| Technique | Role | One-line description | Enabled by |
|---|---|---|---|
| **DD** (Dynamical decoupling) | Suppression | Insert pulse sequences on idle qubits to cancel coherent errors that build up while qubits sit idle | Enable explicitly |
| **PT** (Pauli twirling, gates) | Suppression | Tailor the coherent part of 2Q-gate noise into an effectively stochastic Pauli channel, which also lets downstream methods (PEA, PEC) assume Pauli-channel noise | Level 2 |
| **TREX** (Twirled readout error extinction) | Mitigation | Twirl measurements with random X gates so the readout-error transfer matrix becomes diagonal and can be inverted by rescaling | Level 1 (default) |
| **ZNE** (Zero-noise extrapolation) | Mitigation | Amplify noise at several controlled factors (e.g. by gate folding) and extrapolate the expectation value back to the zero-noise limit (not guaranteed unbiased) | Level 2 |
| **PEA** (Probabilistic error amplification) | Mitigation | Replace ZNE's gate-folding amplification with one driven by a *learned* Pauli-Lindblad noise model, for more accurate noise scaling (still using ZNE's extrapolation step) | Set explicitly (with ZNE) |
| **PEC** (Probabilistic error cancellation) | Mitigation | Sample anti-noise circuits whose ensemble average inverts the learned noise channel, giving an unbiased estimate at the cost of a sampling overhead $\gamma^2$ | Set explicitly |

By default Runtime runs at **resilience level=1**, which enables TREX only; see the [Configure error mitigation](https://docs.quantum.ibm.com/guides/configure-error-mitigation) guide for what each level turns on. The next section looks at each technique's options and behavior in detail.


### 1.1.2  The methods in detail

This section expands each technique from the table in section 1.1.1: what it does, the options that control it, and when it helps. Figures and full explanations are from the official [Error mitigation and suppression techniques](https://docs.quantum.ibm.com/guides/error-mitigation-and-suppression-techniques) guide.

#### Dynamical decoupling (DD) — *suppression*
Idle qubits pick up coherent errors while they wait for other qubits to finish. DD inserts a sequence of pulses on those idle qubits that multiplies out to the identity, so the intended computation is unchanged, while the pulses average away the slow coherent drift. It helps most when a circuit has idle gaps; on a densely packed circuit it can do little, or even hurt, because the pulses are themselves imperfect.

![Dynamical decoupling: an XX pulse sequence inserted during a qubit's idle period (IBM Quantum docs).](https://quantum.cloud.ibm.com/docs/images/guides/error-mitigation-explanation/dd.avif)

- `dynamical_decoupling.enable` — turn DD on (off by default).
- `dynamical_decoupling.sequence_type` — which pulse sequence; default `"XX"`, others include `"XpXm"`, `"XY4"`.

#### Pauli twirling (PT) — *suppression*
Twirling (randomized compiling) sandwiches a chosen gate between random single-qubit Paulis picked so the ideal operation is unchanged. Averaged over many random instances, an arbitrary noise channel becomes a stochastic *Pauli* channel: coherent error, which otherwise accumulates quadratically with depth, is reshaped into Pauli noise that grows only linearly. Since most hardware error comes from two-qubit gates, twirling is usually applied to the native 2Q gates, and it is a prerequisite for PEA and PEC, which assume Pauli-channel noise.

![Pauli twirling replaces one circuit with a random ensemble of equivalent twirled circuits (IBM Quantum docs).](https://quantum.cloud.ibm.com/docs/images/guides/error-mitigation-explanation/pauli_twirling.svg)

- `twirling.enable_gates` — twirl the 2Q gate layers.
- `twirling.num_randomizations` — number of random twirled instances.
- `twirling.shots_per_randomization` — shots per instance.

#### Twirled readout error extinction (TREX) — *mitigation*
TREX targets *measurement* error. It twirls the readout by randomly inserting an X before a measurement and flipping the classical bit back afterward. This, with readout error present, diagonalizes the readout-error transfer matrix so it can be inverted by a simple rescaling. The rescaling factor is learned by benchmarking random circuits prepared in the zero state, at the cost of a few calibration circuits. This is the one method on by default (_resilience level 1_).

![Measurement twirling: an X gate before the measurement and a classical bit-flip after are equivalent to a plain measurement when noiseless (IBM Quantum docs).](https://quantum.cloud.ibm.com/docs/images/guides/error-mitigation-explanation/measurement_twirling.svg)

- `resilience.measure_mitigation` — turn TREX on.
- `resilience.measure_noise_learning.num_randomizations`, `...shots_per_randomization` — control measurement-noise learning.

#### Zero-noise extrapolation (ZNE) — *mitigation*
ZNE runs the circuit at several *amplified* noise levels and extrapolates the expectation value back to zero noise. Runtime amplifies noise by digital gate folding. This replaces a unitary $U$ with $U U^\dagger U$ (and longer foldings) to multiply its noise by a known factor and then fits the results with a chosen functional form. It often improves accuracy but is *not* guaranteed unbiased. The default samples three noise factors, for roughly a 3× overhead.

![Zero-noise extrapolation: digital gate folding amplifies noise; results are extrapolated to the zero-noise limit (IBM Quantum docs).](https://quantum.cloud.ibm.com/docs/images/guides/error-mitigation-explanation/zne.avif)

- `resilience.zne_mitigation` — turn ZNE on.
- `resilience.zne.noise_factors` — amplification factors (default `(1, 3, 5)`).
- `resilience.zne.extrapolator` — fit form (e.g. linear, exponential).

#### Probabilistic error amplification (PEA) — *mitigation*
PEA is a more accurate amplifier *for* ZNE. Instead of gate folding, it first **learns** the twirled Pauli-Lindblad noise model of each entangling layer, then amplifies by probabilistically injecting single-qubit noise drawn from that learned model — a faithful scaling of the actual device noise rather than a synthetic one. The amplified results still feed ZNE's extrapolation, so PEA is switched on top of ZNE. For utility-scale experiments it is often the best amplifier choice.

- `resilience.zne_mitigation = True` together with `resilience.zne.amplifier = "pea"`.

#### Probabilistic error cancellation (PEC) — *mitigation*
PEC also learns the noise model, but instead of amplifying it, it *inverts* it: it samples an ensemble of "anti-noise" circuits whose average implements the inverse noise channel, giving an **unbiased** estimate. The price is a sampling overhead that scales as $\gamma^2$ and rises quickly with circuit depth and total noise — which is why PEC is practical only for circuits of modest size (we return to this cost in 1.3). In Chapter 3, we will implement PEC using _Samplomatic_ and introduce shaded lightcones (SLC) which is an evolution of PEC.

- `resilience.pec_mitigation` — turn PEC on.
- `resilience.pec.max_overhead` — cap on the sampling overhead $\gamma^2$; Runtime backs off if the cap would be exceeded.

__Note__: ZNE/PEA and PEC are *opposing* strategies — amplify the noise versus cancel it — and cannot be enabled on the **same** `EstimatorOptions`.

<div class="alert alert-block alert-success">

**Exercise 1 — Configure each method.** 

Using the methods and attribute paths from section 1.1.2, configure each of the five on its own `EstimatorOptions` object, then collect them into a dictionary `options_dict` with keys `"dd"`, `"pt"`, `"trex"`, `"zne"`, `"pec"`. Each method is checked separately — all five must be correct to pass, and the grader reports any that are not.

- **`dd_options`** — Dynamical decoupling: `enable = True`, `sequence_type = "XY4"`
- **`pt_options`** — Pauli twirling: `enable_gates = True`, `strategy = "active"`, `num_randomizations = 32`
- **`trex_options`** — TREX: `measure_mitigation = True`
- **`zne_options`** — ZNE with PEA: `zne_mitigation = True`, `noise_factors = [1, 3, 5]`, `amplifier = "pea"`
- **`pec_options`** — PEC: `pec_mitigation = True`, `max_overhead = 100`

</div>

In [ ]:
dd_options = EstimatorOptions()
#TODO: YOUR CODE HERE #

pt_options = EstimatorOptions()
#TODO: YOUR CODE HERE #

trex_options = EstimatorOptions()
#TODO: YOUR CODE HERE #

zne_options = EstimatorOptions()
#TODO: YOUR CODE HERE #

pec_options = EstimatorOptions()
#TODO: YOUR CODE HERE #

options_dict = {
    "dd": dd_options,
    "pt": pt_options,
    "trex": trex_options,
    "zne": zne_options,
    "pec": pec_options,
}


In [ ]:
# grade your answer
grade_lab3_ex1(options_dict)

## 1.2  The algebra underneath: dressed layers and commutation

The options in section 1.1 are applied __once__ to the whole circuit. Several of them share a common underlying operation: wrapping a gate with single-qubit Paulis on either side, chosen so that the gate's logical action is unchanged while the noise attached to it is transformed. Gate twirling uses this operation directly. ZNE/PEA and PEC use it indirectly, through the Pauli noise model they learn. A gate wrapped this way is called a **dressed** gate.

In this section, we explain what it means to _dress a gate_. This will help you understand why the error-mitigation switches in section 1.1 behave the way they do. It is also the operation the finer, box-by-box tools in the Samplomatic framework uses. You will see in Chapter 2 how this framework enables us to move beyond the _whole-circuit switch_ approach we have seen so far.

A **dressed layer** is a two-qubit unitary wrapped by single-qubit unitaries on either side. We introduce it first in general, then specialize to dressing a CZ with Paulis — the setting of every two-qubit layer we work with later. The thread holding the section together is the *commutation relation* (Clifford conjugation) between a Pauli and the bare gate: it is what gives the dressing table its structure, and it is the same operation Samplomatic later uses, box by box, to propagate virtual Paulis through a circuit.

### 1.2.1  Dressing a unitary

For any $n$-qubit unitary $U$, a *dressed* version is

$$\tilde{U} \;=\; V_\text{out}\, U\, V_\text{in},$$

where $V_\text{in}$ and $V_\text{out}$ are products of single-qubit unitaries (one per qubit). When the dressing satisfies $V_\text{out}\,U\,V_\text{in} = U$ up to a global phase we call it *invariant*: the logical action of $U$ is unchanged, but any noise channel attached to $U$ gets *twirled* by $V_\text{in}$ and $V_\text{out}$ ([Wallman & Emerson, *Noise tailoring for scalable quantum computation via randomized compiling*, PRA 94, 2016](https://journals.aps.org/pra/abstract/10.1103/PhysRevA.94.052325)). Two questions follow: which pairs $(V_\text{in}, V_\text{out})$ leave $U$ invariant, and how to enumerate them systematically.

The case we will work with throughout the lab is the one in which $V_\text{in}$ and $V_\text{out}$ are Paulis and $U$ is a Clifford — the setting of the two-qubit gates on IBM hardware (CZ, CX, ECR). Two helpers from `qiskit.quantum_info` make it concrete: `Pauli("IX").evolve(U, frame='s')` returns $U\,P\,U^\dagger$ with the correct sign, and `Pauli.equiv(other)` compares two Paulis up to overall phase — the "invariant up to global phase" notion we need.

In [ ]:
def dress(U: QuantumCircuit, v_in: str, v_out: str) -> QuantumCircuit:
    """Build the circuit  V_out . U . V_in  as a QuantumCircuit.
    """
    n = U.num_qubits
    assert not v_in.startswith(("+", "-", "i")), (
        f"v_in must be an unsigned Pauli string, got {v_in!r}"
    )
    assert not v_out.startswith(("+", "-", "i")), (
        f"v_out must be an unsigned Pauli string, got {v_out!r}"
    )
    assert len(v_in) == len(v_out) == n, "Pauli string length must match U's num_qubits"
    qc = QuantumCircuit(n)
    # Qiskit string 'q_{n-1}...q_0' — reverse to iterate in qubit order.
    for q, p in enumerate(reversed(v_in)):
        if p != "I":
            getattr(qc, p.lower())(q)
    qc.compose(U, inplace=True)
    for q, p in enumerate(reversed(v_out)):
        if p != "I":
            getattr(qc, p.lower())(q)
    return qc


def is_invariant(U: QuantumCircuit, v_in: str, v_out: str) -> bool:
    """True iff  V_out . U . V_in  equals  U  up to a global phase.

    Takes *unsigned* Pauli strings v_in, v_out. The invariance condition is
    U V_in U† = ±V_out; the ± becomes a global phase on the dressed operator,
    so it does not affect invariance. We use `Pauli.equiv`, which is
    phase-insensitive, to compare.
    """
    return Pauli(v_in).evolve(U, frame='s').equiv(Pauli(v_out))

### 1.2.2  CZ Example

We specialize to $U = \mathrm{CZ}$ because that is the two-qubit gate used throughout the lab — the Ising circuit in section 2.6.1 is built entirely from CZ gates. After verifying by direct calculation that a specific Pauli dressing leaves CZ invariant, we propagate all 16 two-qubit Paulis through CZ to build the full map $V_\text{in} \mapsto V_\text{out}$. There is exactly one valid $V_\text{out}$ (up to sign) for each $V_\text{in}$.

Two distinct pieces of information live in the resulting table we print below and they should not be conflated. The *dressing map* records which **unsigned** Pauli to physically apply as $V_\text{out}$, and is what `dress()` consumes. 

The __algebraic sign__, which defines whether $\mathrm{CZ}\cdot V_\text{in}\cdot \mathrm{CZ}$ equals $+V_\text{out}$ or $-V_\text{out}$ does **not** affect gate invariance (the sign is absorbed as a global phase), but it **does** matter for the propagation rules in section 1.2.3.

We will see that the table itself has a clean structure: $Z$ factors commute through CZ untouched, an $X$ or $Y$ factor on one qubit picks up a $Z$ on the **other** qubit, and the pair $XY \leftrightarrow YX$ picks up a minus sign — visible in the rightmost (sign) column of the printout below.

In [ ]:
# Bare CZ layer
cz_layer = QuantumCircuit(2, name="CZ")
cz_layer.cz(0, 1)
cz_layer.draw("mpl")

In [ ]:
# apply X on qubit 0 before CZ, and X on qubit 0 + Z on qubit 1 after.

print("V_in='IX', V_out='ZX' leaves CZ invariant?",
      is_invariant(cz_layer, "IX", "ZX"))

In [ ]:
# Build two related objects from the Pauli algebra:
#   cz_twirl_map       : V_in -> V_out  (UNSIGNED — what to apply as a dressing)
#   cz_commutation_map : V_in -> (sign, V_out)  (signed — for propagation rules)

PAULIS = ["I", "X", "Y", "Z"]


def split_sign(label: str) -> tuple[int, str]:
    """Split a signed Pauli label into (+/-1, unsigned_label)."""
    if label.startswith("-"):
        return -1, label[1:]
    if label.startswith("+"):
        return +1, label[1:]
    return +1, label


cz_twirl_map       = {}
cz_commutation_map = {}

for p1 in PAULIS:         # Pauli on qubit 1 (leftmost in string)
    for p0 in PAULIS:     # Pauli on qubit 0 (rightmost in string)
        v_in   = p1 + p0
        signed = Pauli(v_in).evolve(cz_layer, frame="s").to_label()
        sign, bare = split_sign(signed)
        cz_twirl_map[v_in]       = bare
        cz_commutation_map[v_in] = (sign, bare)

print(f"Total Pauli dressings for CZ: {len(cz_twirl_map)}  (expected 16)")
print()
print(f"  {'V_in':<6}{'V_out (apply)':<16}{'algebraic sign'}")
print(f"  {'-'*6}{'-'*16}{'-'*14}")
for v_in, v_out in cz_twirl_map.items():
    sign, _ = cz_commutation_map[v_in]
    sign_str = "+" if sign > 0 else "−"
    print(f"  {v_in:<6}{v_out:<16}{sign_str}")

### 1.2.3  Commutation: how a Pauli propagates through a Clifford

The structure of the table in section 1.2.2 is no accident. For any Pauli $P$ and Clifford $U$,

$$U\,P\,U^\dagger \;=\; \pm\,P',$$

where $P'$ is another Pauli. This is the defining property of the Clifford group. Finding $V_\text{out}$ given $V_\text{in}$ amounts to propagating $V_\text{in}$ through $U$ and reading off the result, sign included. This is the operation Samplomatic uses in chapter 2 every time it propagates a virtual Pauli across a gate box. `Pauli.evolve(U, frame='s')` is the Qiskit helper that implements it, and `.to_label()` returns a signed Pauli string such as `"-YX"`.

Running the same helper on CX produces a structurally similar table with an asymmetry from CX's control/target roles.

In [ ]:
# The same Pauli.evolve(..., frame="s") helper, applied to two different gates.
# The propagation rule is fixed by the gate, so CZ and CX give different tables.
cx_layer = QuantumCircuit(2, name="CX")
cx_layer.cx(0, 1)  # control = q0, target = q1

reps = ["IX", "XI", "IZ", "ZI", "IY", "YI"]

print("CZ . P . CZ  (symmetric in the two qubits):")
for p in reps:
    print(f"  CZ . {p} . CZ  =  {Pauli(p).evolve(cz_layer, frame='s').to_label()}")

print("\nCX . P . CX  (control = q0, target = q1):")
for p in reps:
    print(f"  CX . {p} . CX  =  {Pauli(p).evolve(cx_layer, frame='s').to_label()}")

The two tables above differ because the propagation rule is fixed by the gate. CZ is symmetric: a $Z$ on either qubit passes through, an $X$ or $Y$ picks up a $Z$ on the other qubit. CX is not: an $X$ on the target qubit and a $Z$ on the control qubit each spread to both qubits, but an $X$ on the control qubit and a $Z$ on the target qubit pass through unchanged. 

The procedure is the same in both cases: conjugate the Pauli by the gate and read off the signed result. This is the operation every twirling-based method in the lab is built on, from the `twirling.enable_gates` switch in section 1.1.2 to Samplomatic's per-box `Twirl` annotation we will introduce in Chapter 2.

What separates those methods is the scale at which the operation is applied. An `EstimatorOptions` switch applies it once to the whole circuit — the coarsest option, and one that doesn't work for some mitigation tasks, such as one which require treating individual layers or observables differently. Secton 1.3 sets out what the Runtime options can and cannot do, along with their resource cost, which is what motivates the finer tools used in the rest of the lab.

## 1.3  Why we need structure-aware mitigation

In the previous sections, we showed what the `EstimatorOptions` switch can do and the algebra underneath it. This section looks at where a whole-circuit switch falls short, the error mitigation tasks it cannot express, and the resource cost it incurs as circuits grow. We do this with Samplomatic: the per-box tool that the rest of the lab is built on.

#### What the Runtime options cannot do

Runtime options expose **whole-circuit switches** like "insert DD on every idle window" or "twirl every 2Q gate". Each is applied by a single circuit-wide policy, which is sufficient for many small workloads. PEA and PEC do learn a noise model *per entangling layer* internally, but the API exposes no per-layer controls: you cannot inspect a layer's learned noise, override it, or apply a different strategy to different layers. Another capability is out of reach entirely: rewriting the observable itself to absorb learned noise rather than mitigating the circuit (something we will see in Chapter 3 with __propagated noise absorption__ (PNA)). 

Chapter 2 introduces the tools that provide this finer control: the *box* — a slice of the circuit annotated with directives such as `Twirl`, `InjectNoise`, or `ChangeBasis` — together with `NoiseLearnerV3` and `Executor`, the box-aware counterparts to `Sampler` and `Estimator`.

#### The hidden cost: sampling overhead

Turning these switches on is not free, and the bill is paid in **shots**, not qubits. Error mitigation trades a _space_ overhead for a _time_ overhead: instead of extra physical qubits (as in error correction), it runs many noisy circuits and post-processes them. Part of this cost is a plain multiplier you can read off the options, like in ZNE's `noise_factors`, `twirling.num_randomizations`, and TREX's readout-calibration batch. For example, three ZNE factors with 32 randomizations already runs on the order of `3 × 32` variants relative to a single unmitigated run.

What you *cannot* read off from the options is something that limits utility-scale circuits: the __sampling overhead__ of probabilistic methods grows *exponentially*. For PEC, the runtime $J$ is given by
$$J = \bar{\gamma}^{\,nd}\,\beta^{d},$$
where $n$ is the number of qubits and $d$ is the depth. The base $\bar{\gamma}$ is set by the *learned* noise model. This can be thought of as a quality metric: a lower gate error gives a smaller $\bar{\gamma}$. $\beta$ is the time per circuit layer (a speed metric). Because $\bar{\gamma}$ enters exponentially, even a small reduction in it, i.e. by having a lower gate error, will reduce the size of $\bar{\gamma}^{nd}$ dramatically. 

<img src="https://research-website-prod-cms-uploads.s3.us.cloud-object-storage.appdomain.cloud/New_Fig_Gamma_Epsilon2_Tex_100q_trotter_a712254dca.jpg" width="450" alt="Estimated PEC circuit overhead for 100-qubit Ising Trotter circuits">

*Fig. 1: Estimating the PEC circuit overhead for 100 qubit Trotterization circuits of depth 400 and 4000, representing the time evolution of an Ising spin chain. Dotted red line indicates 1 day of runtime assuming a fixed 1kHz sampling rate. Reproduced from K. Temme, E. van den Berg, A. Kandala, and J. Gambetta, "[Error mitigation is the path to quantum computing usefulness](https://www.ibm.com/quantum/blog/gammabar-for-quantum-advantage)," IBM Quantum blog, 2022. © IBM.*

The figure above is for the same kind of circuit this lab uses — an Ising Hamiltonian Trotter evolution (we will introduce this in section 2.6). The figure shows the overhead dropping by orders of magnitude as the two-qubit gate error falls. The whole-circuit switches available to us with `EstimatorOptions` give no way to see or act on the per-layer noise that sets $\bar{\gamma}$. This is the gap the rest of the lab fills: Chapter 2 *measures* the per-layer noise with `NoiseLearnerV3`, and Chapter 3 *reduces* the overhead by treating layers and observables individually (PNA and SLC).